 Task 01 – Customer Churn Prediction




In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 1: Load the dataset
df = pd.read_csv('/content/dataset-1.csv')

# Convert Churn column to binary
if 'Churn' in df.columns:
    df['Churn'] = df['Churn'].astype(int)

# Separate features (X) and target (y)
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

# Step 2: Split the data FIRST before any transformations
# First split: Extract the test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# Second split: Divide the remaining data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

print("Data successfully split into Train, Validation, and Test sets.")

Data successfully split into Train, Validation, and Test sets.


In [ ]:
# Function to remove duplicate from training data set
def remove_bad_training_rows(X_train_data, y_train_data):
    X_train_clean = X_train_data.drop_duplicates()
    y_train_clean = y_train_data.loc[X_train_clean.index]
    return X_train_clean, y_train_clean

# Apply the cleaning function only to the training set
X_train, y_train = remove_bad_training_rows(X_train, y_train)

print(f"Training data shape after cleaning: {X_train.shape}")

Training data shape after cleaning: (3926, 19)


In [ ]:
# Convert 'TotalCharges' to numeric across all splits, coercing errors to NaN
X_train['TotalCharges'] = pd.to_numeric(X_train['TotalCharges'], errors='coerce')
X_val['TotalCharges'] = pd.to_numeric(X_val['TotalCharges'], errors='coerce')
X_test['TotalCharges'] = pd.to_numeric(X_test['TotalCharges'], errors='coerce')

# Define numeric and categorical feature columns
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

# Pipeline for numerical features (Impute missing with median -> Scale)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features (Impute missing with most frequent -> One-Hot Encode)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Build the final pipeline with the preprocessor and the Random Forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Machine Learning Pipeline successfully created.")

Machine Learning Pipeline successfully created.


In [ ]:
# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Define a function to evaluate the model
def evaluate_model(model, X_eval, y_eval, dataset_name='Test'):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    print(f'--- {dataset_name} Performance ---')
    print(f'Accuracy: {accuracy_score(y_eval, y_pred):.4f}')
    print(f'F1-Score: {f1_score(y_eval, y_pred):.4f}')
    print(f'ROC-AUC:  {roc_auc_score(y_eval, y_proba):.4f}\n')

# Evaluate the model across all three datasets
evaluate_model(pipeline, X_train, y_train, 'Training')
evaluate_model(pipeline, X_val, y_val, 'Validation')
evaluate_model(pipeline, X_test, y_test, 'Test')

--- Training Performance ---
Accuracy: 1.0000
F1-Score: 1.0000
ROC-AUC:  1.0000

--- Validation Performance ---
Accuracy: 0.7825
F1-Score: 0.5306
ROC-AUC:  0.8267

--- Test Performance ---
Accuracy: 0.7967
F1-Score: 0.5544
ROC-AUC:  0.8184



In [ ]:
# Required function to predict churn on new input data
def predict_churn(model, input_data):
    """
    Takes the trained model pipeline and new input data,
    returns binary predictions and churn probabilities.
    """
    predictions = model.predict(input_data)
    probabilities = model.predict_proba(input_data)[:, 1]
    return predictions, probabilities

print("Ready for predictions. Call predict_churn(pipeline, new_data_dataframe) to use.")

Ready for predictions. Call predict_churn(pipeline, new_data_dataframe) to use.
